# 🔬 Breast Cancer Prediction — ANN with PyTorch
Using all 30 features from the Wisconsin Breast Cancer Dataset

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('All libraries imported successfully!')
print(f'PyTorch version: {torch.__version__}')

## 2. Load & Prepare Data

In [ ]:
# ✅ Update this path to where your CSV file is located
df = pd.read_csv('breast-cancer.csv')

# Drop ID column and encode diagnosis (M=1, B=0)
df = df.drop(columns=['id'])
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

print(f'Dataset shape: {df.shape}')
print(f'Malignant (1): {df["diagnosis"].sum()}')
print(f'Benign    (0): {(df["diagnosis"] == 0).sum()}')
df.head()

## 3. Use ALL 30 Features

In [ ]:
# ✅ Using ALL 30 features instead of just 2
feature_columns = [
    'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean',
    'smoothness_mean', 'compactness_mean', 'concavity_mean',
    'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
    'radius_se', 'texture_se', 'perimeter_se', 'area_se',
    'smoothness_se', 'compactness_se', 'concavity_se',
    'concave points_se', 'symmetry_se', 'fractal_dimension_se',
    'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst',
    'smoothness_worst', 'compactness_worst', 'concavity_worst',
    'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst'
]

X = df[feature_columns].values   # shape: (569, 30)
y = df['diagnosis'].values.reshape(-1, 1)  # shape: (569, 1)

print(f'X shape: {X.shape}  →  {X.shape[1]} features')
print(f'y shape: {y.shape}')

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')

## 5. Scale Features
> ✅ Scaler is fitted ONCE on training data only, then saved

In [ ]:
# ✅ Fit scaler ONCE on training data only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)   # ← transform only, never fit on test!

# ✅ Save scaler immediately so it always matches the trained model
pickle.dump(scaler, open('scaler.pkl', 'wb'))
print('Scaler fitted and saved as scaler.pkl')
print(f'X_train scaled shape: {X_train.shape}')

## 6. Convert to PyTorch Tensors

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

print(f'X_train tensor: {X_train_t.shape}')
print(f'X_test  tensor: {X_test_t.shape}')

## 7. Define ANN Model
> ✅ Updated to accept 30 input features, with a wider architecture

In [ ]:
class ANN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),  # 30 → 64
            nn.ReLU(),
            nn.Dropout(0.3),           # prevents overfitting

            nn.Linear(64, 32),         # 64 → 32
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 16),         # 32 → 16
            nn.ReLU(),

            nn.Linear(16, 1),          # 16 → 1
            nn.Sigmoid()               # output between 0 and 1
        )

    def forward(self, x):
        return self.network(x)


# ✅ input_dim=30 because we now use all 30 features
model = ANN(input_dim=X_train_t.shape[1])
print(model)
print(f'\nInput features: {X_train_t.shape[1]}')
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params}')

## 8. Train the Model

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # Adam is better than SGD here

epochs = 300  # fewer epochs needed with Adam

train_losses = []

for epoch in range(epochs):
    model.train()

    y_pred = model(X_train_t)
    loss = criterion(y_pred, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(f'Epoch [{epoch+1}/{epochs}]  Loss: {loss.item():.4f}')

print('\nTraining complete!')

## 9. Evaluate the Model

In [ ]:
model.eval()

with torch.no_grad():
    y_train_pred = (model(X_train_t) >= 0.5).float()
    y_test_pred  = (model(X_test_t)  >= 0.5).float()

accuracy_train = accuracy_score(y_train_t, y_train_pred)
accuracy_test  = accuracy_score(y_test_t,  y_test_pred)

print(f'Train Accuracy : {accuracy_train * 100:.2f}%')
print(f'Test  Accuracy : {accuracy_test  * 100:.2f}%')

print('\nClassification Report (Test Set):')
print(classification_report(
    y_test_t.numpy(),
    y_test_pred.numpy(),
    target_names=['Benign', 'Malignant']
))

## 10. Save the Model
> ✅ Using `torch.save(state_dict)` — the correct PyTorch way

In [ ]:
# ✅ Save model weights properly using state_dict (NOT pickle)
torch.save(model.state_dict(), 'breast_cancer_model.pth')
print('Model saved as breast_cancer_model.pth')

# ✅ Verify it loads correctly
model_check = ANN(input_dim=30)
model_check.load_state_dict(torch.load('breast_cancer_model.pth', map_location='cpu'))
model_check.eval()
print('Model reloaded and verified successfully!')

## 11. Test a Single Prediction
> This is exactly how the Hugging Face app will call the model

In [ ]:
# Example: use the first row from the test set
sample = X_test[0].reshape(1, -1)          # shape (1, 30) — numpy
sample_scaled = scaler.transform(sample)    # already scaled but showing the flow

# Actually let's use raw data to show the full prediction pipeline
raw_sample = X_test[0].reshape(1, -1)       # raw unscaled test sample

# Reload scaler from disk to simulate production
with open('scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)

scaled = loaded_scaler.transform(raw_sample)
tensor = torch.tensor(scaled, dtype=torch.float32)

loaded_model = ANN(input_dim=30)
loaded_model.load_state_dict(torch.load('breast_cancer_model.pth', map_location='cpu'))
loaded_model.eval()

with torch.no_grad():
    prob = loaded_model(tensor).item()

label = 'Malignant' if prob >= 0.5 else 'Benign'
print(f'Prediction : {label}')
print(f'Probability: {prob:.4f}')
print(f'Actual     : {"Malignant" if y_test[0][0] == 1 else "Benign"}')

## ✅ Files ready for Hugging Face deployment
```
breast-cancer-predictor/
├── app.py                      ← Gradio app (create separately)
├── breast_cancer_model.pth     ← Model weights  ✅
├── scaler.pkl                  ← Fitted scaler  ✅
└── requirements.txt            ← Dependencies   ✅
```